# Credit Scoring Fairness Audit with Bias Mitigation
## Complete Master's Project Notebook — German Credit, Home Credit and Taiwan Credit Card Default

**Purpose.** This notebook implements the methodology in the uploaded Master's research proposal from raw-data inspection through predictive modelling, fairness auditing, bias mitigation, accuracy–fairness trade-off analysis, cross-dataset comparison and research-question conclusions.

### Important dataset clarification
The proposal names **FCA Financial Lives Survey + German Credit + Home Credit**. The three data files supplied with this notebook are **German Credit**, **Home Credit**, and **Taiwan Default of Credit Card Clients**. Therefore, this notebook uses the *actual uploaded third dataset (Taiwan)* rather than pretending it is Financial Lives. This preserves data integrity. If the final dissertation still claims FCA Financial Lives was used, the methodology chapter must be updated to match the dataset actually analysed.

### Proposal requirements implemented
The proposal requires: data preprocessing; Logistic Regression, Random Forest and XGBoost; Demographic Parity Difference (DPD); Equalized Odds Difference (EOD); Reweighing; Equalized-Odds post-processing; and performance evaluation using Accuracy, Precision, Recall and F1-score. This notebook also reports ROC-AUC because the target is binary and class imbalance is present.

### Outcome convention
For all three datasets the modelling target is coded consistently:

- **0 = good/non-default**
- **1 = bad/default/payment difficulty**

Predictive metrics evaluate the **default class (1)**. Fairness metrics are presented using the **favourable decision (approval/non-default = 1)** by complementing predictions. This makes Demographic Parity directly interpretable as a difference in favourable decision/approval rates.

### Protected attributes
Two demographic attributes are audited where genuinely available in all three datasets:

- **Gender/Sex**
- **Age group** (derived from the original age variable only for fairness grouping; the raw observation itself is not changed)

Protected variables are retained for auditing but **excluded from model predictors**. This avoids direct use of protected characteristics while still allowing the study to detect possible proxy-driven disparities, which is consistent with the research proposal's discussion of proxy discrimination.

## 1. Environment Setup

**What are we doing?** Importing the libraries needed for data handling, modelling, fairness analysis and visualisation.

**Why?** The project requires conventional ML plus fairness-specific metrics and post-processing.

**Viva point.** `fairlearn` is used for fairness metrics and Equalized-Odds post-processing. Reweighing is implemented transparently from the standard group–outcome weighting formula so its logic can be explained in the viva.

In [ ]:
# Kaggle normally includes scikit-learn, xgboost and matplotlib.
# The legacy Taiwan .xls file requires xlrd; Fairlearn is used for fairness analysis.
import sys, subprocess, importlib.util

REQUIRED = {
    "xlrd": "xlrd",
    "fairlearn": "fairlearn",
    "xgboost": "xgboost"
}
for module_name, pip_name in REQUIRED.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import os, glob, warnings, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve
)
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.postprocessing import ThresholdOptimizer
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
RANDOM_STATE = 42
TEST_SIZE = 0.20
CALIBRATION_SIZE_WITHIN_TRAIN = 0.15

# ---------------- SPEED SETTINGS ----------------
# FAST_MODE is recommended for Kaggle development/version saving.
# It uses a reproducible stratified research sample for very large datasets.
# No synthetic observations are created and the original target proportions are preserved.
FAST_MODE = True
MAX_ROWS = {"German": None, "Home Credit": 60000, "Taiwan": None}
SHOW_EXTRA_PLOTS = False   # False = only the most useful consolidated plots

print("Environment ready. FAST_MODE =", FAST_MODE)

## Kaggle Fast Mode
This optimized version keeps the full research methodology but reduces runtime. **Home Credit is analysed using a reproducible stratified sample of 60,000 genuine observations** when `FAST_MODE=True`. No synthetic data is created. Set `FAST_MODE=False` only for the final full-data run if required. Repetitive plots are also suppressed to reduce Kaggle version size.


## 2. Locate and Load the Three Uploaded Datasets

**What are we doing?** Searching common Kaggle and local paths so the notebook can run without manually editing paths.

**Why?** Kaggle changes the parent folder name when a dataset is uploaded. Searching recursively by the actual filenames makes the notebook portable.

**Data-integrity rule.** The notebook does not fabricate records or force the datasets into identical schemas. Each dataset has its own loader and preparation function.

In [ ]:
def find_uploaded_file(filename_candidates):
    roots = ["/kaggle/input", "/mnt/data", "."]
    for root in roots:
        if not os.path.exists(root):
            continue
        for candidate in filename_candidates:
            matches = glob.glob(os.path.join(root, "**", candidate), recursive=True)
            if matches:
                return matches[0]
    raise FileNotFoundError(
        "Could not find any of these files: " + ", ".join(filename_candidates)
    )

GERMAN_PATH = find_uploaded_file([
    "german_credit_data(3).csv", "german_credit_data.csv"
])
HOME_PATH = find_uploaded_file([
    "Home_Credit_Default_Risk.csv",
    "Home_Credit_Default_Risk_Final_Combined.csv"
])
TAIWAN_PATH = find_uploaded_file([
    "default of credit card clients(1).xls",
    "default of credit card clients.xls"
])

print("German:", GERMAN_PATH)
print("Home  :", HOME_PATH)
print("Taiwan:", TAIWAN_PATH)

In [ ]:
def load_taiwan_xls(path):
    """Load the UCI Taiwan legacy .xls robustly.
    The common file contains a title row above the real header, so header=1 is tried first.
    The function verifies the target name instead of assuming the read succeeded.
    """
    candidates = []
    for header in [1, 0]:
        try:
            d = pd.read_excel(path, header=header, engine="xlrd")
            candidates.append(d)
            normalized = [str(c).strip().lower() for c in d.columns]
            if any("default payment next month" in c for c in normalized):
                return d
        except Exception as e:
            last_error = e
    if candidates:
        return candidates[0]
    raise last_error

german_raw = pd.read_csv(GERMAN_PATH)
home_raw = pd.read_csv(HOME_PATH)
taiwan_raw = load_taiwan_xls(TAIWAN_PATH)

print("German shape:", german_raw.shape)
print("Home shape  :", home_raw.shape)
print("Taiwan shape:", taiwan_raw.shape)

## 3. Raw Data Inspection

**What are we doing?** Inspecting dimensions, columns, data types, duplicates, missing values and target distributions *before* cleaning.

**Why?** The three datasets have different structures. Inspection prevents incorrect assumptions and documents the starting point of the analysis.

**What does the output mean?** Large missing-value counts indicate variables requiring imputation; duplicate rows would require investigation; target distributions show whether class imbalance is present.

In [ ]:
def raw_inspection(name, df, target_hint=None):
    print("\n" + "="*90)
    print(name.upper())
    print("="*90)
    print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")
    print(f"Exact duplicate rows: {df.duplicated().sum():,}")
    print("\nColumn names:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes.value_counts())
    print("\nTop 15 columns by missing values:")
    miss = df.isna().sum().sort_values(ascending=False)
    print(pd.DataFrame({"missing_n": miss.head(15), "missing_%": (100*miss/len(df)).head(15).round(2)}))
    if target_hint is not None and target_hint in df.columns:
        print("\nTarget distribution:")
        print(df[target_hint].value_counts(dropna=False))
    print("\nFirst 3 rows:")
    display(df.head(3))

raw_inspection("German Credit", german_raw, "Risk")
raw_inspection("Home Credit", home_raw, "TARGET")

# Locate Taiwan target from the actual loaded columns.
taiwan_target_candidates = [c for c in taiwan_raw.columns if "default payment next month" in str(c).strip().lower()]
TAIWAN_TARGET_RAW = taiwan_target_candidates[0] if taiwan_target_candidates else taiwan_raw.columns[-1]
raw_inspection("Taiwan Credit Card Default", taiwan_raw, TAIWAN_TARGET_RAW)

In [ ]:
def inspection_interpretation(name, df, target_col):
    miss_cells = int(df.isna().sum().sum())
    dup = int(df.duplicated().sum())
    target_counts = df[target_col].value_counts(dropna=False)
    print(f"{name}: {len(df):,} observations and {df.shape[1]:,} variables were inspected.")
    print(f"There are {dup:,} exact duplicate rows and {miss_cells:,} missing cells in the raw file.")
    print("Target counts:", target_counts.to_dict())
    if len(target_counts) == 2:
        minority = target_counts.min() / target_counts.sum()
        print(f"The minority target class represents {minority:.1%} of observations, so stratified splitting and class-sensitive metrics are important.")

inspection_interpretation("German Credit", german_raw, "Risk")
inspection_interpretation("Home Credit", home_raw, "TARGET")
inspection_interpretation("Taiwan", taiwan_raw, TAIWAN_TARGET_RAW)

## 4. Dataset-Specific Preparation and Target Coding

### Methodological decisions

**Missing values.** Numerical variables are imputed with the **training-set median**, which is robust to outliers. Categorical variables are imputed with an explicit **"Missing"** category so missingness is not silently treated as an observed category. Imputation is fitted only on training data inside a pipeline, preventing test-data leakage.

**Encoding.** Nominal categorical variables are one-hot encoded. Unknown test categories are ignored safely.

**Scaling.** Numerical variables are standardised for Logistic Regression. The same preprocessing pipeline is used consistently for all three model families for comparability, although tree models do not inherently require scaling.

**Duplicates.** Exact duplicate rows are removed only when they are genuinely duplicate observations. Identifier columns are excluded from prediction because identifiers do not represent applicant risk.

**Class imbalance.** No synthetic observations (for example SMOTE) are created. The original observations are preserved. Splits are stratified by target, and Precision/Recall/F1/ROC-AUC are reported alongside Accuracy so imbalance cannot be hidden by accuracy alone.

**Protected variables.** Gender/Sex and raw age are excluded from prediction but retained separately for fairness auditing. Age is grouped only for fairness comparison; this does not change the original age values.

In [ ]:
def age_groups(age_series):
    # Broad adult age bands used only as audit groups, not as legal categories.
    return pd.cut(
        pd.to_numeric(age_series, errors="coerce"),
        bins=[-np.inf, 34, 49, np.inf],
        labels=["18-34", "35-49", "50+"],
        right=True
    ).astype("object")


def prepare_german(df):
    d = df.copy()
    d = d.drop_duplicates().copy()
    d = d.drop(columns=[c for c in d.columns if str(c).startswith("Unnamed")], errors="ignore")
    assert "Risk" in d.columns and "Sex" in d.columns and "Age" in d.columns
    d["TARGET"] = d["Risk"].astype(str).str.lower().map({"good":0, "bad":1})
    if d["TARGET"].isna().any():
        raise ValueError("German Risk contains unexpected values: " + str(d["Risk"].unique()))
    protected = pd.DataFrame(index=d.index)
    protected["Gender"] = d["Sex"].astype("object")
    protected["Age_Group"] = age_groups(d["Age"])
    # Job codes are category labels rather than continuous measurements.
    if "Job" in d.columns:
        d["Job"] = d["Job"].astype(str)
    X = d.drop(columns=["Risk", "TARGET", "Sex", "Age"], errors="ignore")
    y = d["TARGET"].astype(int)
    return X, y, protected, d


def prepare_home(df):
    d = df.copy().drop_duplicates().copy()
    assert "TARGET" in d.columns and "CODE_GENDER" in d.columns and "DAYS_BIRTH" in d.columns
    d["TARGET"] = pd.to_numeric(d["TARGET"], errors="raise").astype(int)
    age = -pd.to_numeric(d["DAYS_BIRTH"], errors="coerce") / 365.25
    protected = pd.DataFrame(index=d.index)
    protected["Gender"] = d["CODE_GENDER"].replace({"XNA": np.nan}).astype("object")
    protected["Age_Group"] = age_groups(age)
    X = d.drop(columns=["TARGET", "SK_ID_CURR", "CODE_GENDER", "DAYS_BIRTH"], errors="ignore")
    y = d["TARGET"].astype(int)
    return X, y, protected, d


def prepare_taiwan(df):
    d = df.copy().drop_duplicates().copy()
    # Standardise header text but preserve the underlying observations.
    d.columns = [str(c).strip() for c in d.columns]
    target_candidates = [c for c in d.columns if "default payment next month" in c.lower()]
    if not target_candidates:
        raise ValueError("Taiwan target column 'default payment next month' was not found after loading.")
    target_col = target_candidates[0]
    required = ["SEX", "AGE"]
    missing_required = [c for c in required if c not in d.columns]
    if missing_required:
        raise ValueError(f"Taiwan required columns missing: {missing_required}. Actual columns: {d.columns.tolist()}")
    d["TARGET"] = pd.to_numeric(d[target_col], errors="raise").astype(int)
    sex_map = {1:"Male", 2:"Female", "1":"Male", "2":"Female"}
    protected = pd.DataFrame(index=d.index)
    protected["Gender"] = d["SEX"].map(sex_map).astype("object")
    protected["Age_Group"] = age_groups(d["AGE"])
    # EDUCATION and MARRIAGE are coded categories in the source dataset.
    for c in ["EDUCATION", "MARRIAGE"]:
        if c in d.columns:
            d[c] = d[c].astype(str)
    X = d.drop(columns=[target_col, "TARGET", "ID", "SEX", "AGE"], errors="ignore")
    y = d["TARGET"].astype(int)
    return X, y, protected, d

DATASETS = {}
for name, raw, prep in [
    ("German", german_raw, prepare_german),
    ("Home Credit", home_raw, prepare_home),
    ("Taiwan", taiwan_raw, prepare_taiwan)
]:
    X, y, protected, cleaned = prep(raw)
    # Optional speed mode for the large Home Credit table.
    # This is stratified random sampling from real observations, not synthetic data.
    limit = MAX_ROWS.get(name) if FAST_MODE else None
    if limit is not None and len(y) > limit:
        sample_idx, _ = train_test_split(
            np.arange(len(y)), train_size=limit, stratify=y, random_state=RANDOM_STATE
        )
        sample_idx = np.sort(sample_idx)
        X = X.iloc[sample_idx].reset_index(drop=True)
        y = y.iloc[sample_idx].reset_index(drop=True)
        protected = protected.iloc[sample_idx].reset_index(drop=True)
        cleaned = cleaned.iloc[sample_idx].reset_index(drop=True)
        print(f"{name}: FAST_MODE uses {limit:,} real stratified observations from the original dataset.")

    DATASETS[name] = {"X":X, "y":y, "protected":protected, "cleaned":cleaned}
    print(f"{name}: X={X.shape}, target={y.value_counts().to_dict()}")
    print("Protected groups:")
    for p in protected.columns:
        print(" ", p, protected[p].value_counts(dropna=False).to_dict())

## 5. Exploratory Data Analysis (EDA)

The EDA is deliberately focused rather than repetitive. It shows:

1. Target distribution and class imbalance.
2. Default rate across protected groups.
3. Missingness in the most incomplete variables.
4. Correlation of the strongest numerical variables with the target.

**Viva point.** Correlation is descriptive, not proof of causation or unfairness. Fairness is evaluated later using model outcomes and dedicated metrics.

In [ ]:
def plot_target_distribution(name, y):
    counts = y.value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6,4))
    ax.bar(["0 = Good/Non-default", "1 = Default"], counts.values)
    ax.set_title(f"{name}: Target Distribution")
    ax.set_ylabel("Number of observations")
    for i,v in enumerate(counts.values):
        ax.text(i, v, f"{v:,}\n({v/len(y):.1%})", ha="center", va="bottom")
    plt.tight_layout(); plt.show()
    minority = counts.min()/counts.sum()
    print(f"Interpretation: The minority class is {minority:.1%} of {name}. Accuracy alone would therefore be insufficient; Recall, F1 and ROC-AUC are required.")


def plot_protected_default_rates(name, y, protected):
    for attr in protected.columns:
        temp = pd.DataFrame({"TARGET":y, attr:protected[attr]}).dropna()
        rates = temp.groupby(attr, observed=True)["TARGET"].agg(["mean","count"]).sort_values("mean")
        fig, ax = plt.subplots(figsize=(7,4))
        ax.bar(rates.index.astype(str), rates["mean"])
        ax.set_title(f"{name}: Observed Default Rate by {attr}")
        ax.set_ylabel("Observed default rate")
        ax.tick_params(axis="x", rotation=25)
        plt.tight_layout(); plt.show()
        gap = rates["mean"].max()-rates["mean"].min()
        print(f"Interpretation: The raw observed default-rate range across {attr} groups is {gap:.3f}. This is descriptive population disparity, not yet evidence that a model itself is unfair.")
        display(rates)


def plot_missingness(name, X, top_n=12):
    miss = X.isna().mean().sort_values(ascending=False).head(top_n)
    if miss.max() == 0:
        print(f"{name}: no missing predictor values detected.")
        return
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(miss.index.astype(str), 100*miss.values)
    ax.set_title(f"{name}: Highest Predictor Missingness")
    ax.set_ylabel("Missing (%)")
    ax.tick_params(axis="x", rotation=70)
    plt.tight_layout(); plt.show()
    print(f"Interpretation: The most incomplete predictor is '{miss.index[0]}' at {miss.iloc[0]:.1%} missing. Missing values are imputed inside the training pipeline rather than deleting applicants.")


def plot_top_numeric_correlations(name, X, y, top_n=12):
    numeric = X.select_dtypes(include=np.number)
    if numeric.empty:
        return
    temp = numeric.copy()
    temp["TARGET"] = y
    corr = temp.corr(numeric_only=True)["TARGET"].drop("TARGET").dropna()
    if corr.empty:
        return
    top = corr.reindex(corr.abs().sort_values(ascending=False).head(top_n).index).sort_values()
    fig, ax = plt.subplots(figsize=(8,5))
    ax.barh(top.index.astype(str), top.values)
    ax.set_title(f"{name}: Numerical Variables Most Correlated with Default")
    ax.set_xlabel("Pearson correlation with TARGET")
    plt.tight_layout(); plt.show()
    print(f"Interpretation: '{corr.abs().idxmax()}' has the strongest absolute linear correlation with default among the numerical predictors ({corr.loc[corr.abs().idxmax()]:.3f}). Correlation alone does not determine model importance or causality.")

for name, obj in DATASETS.items():
    print("\n###", name)
    plot_target_distribution(name, obj["y"])
    if SHOW_EXTRA_PLOTS:
        plot_protected_default_rates(name, obj["y"], obj["protected"])
        plot_missingness(name, obj["X"])
        plot_top_numeric_correlations(name, obj["X"], obj["y"])


## 6. Train / Calibration / Test Split

**What are we doing?** Each dataset is split into a final test set (20%) and a development set (80%). The development portion is then divided into model-training and calibration subsets.

**Why?** The untouched test set gives an unbiased final evaluation. The separate calibration subset lets Equalized-Odds post-processing learn group-specific decision rules **without using the test set**.

**Why stratification?** Stratification preserves approximately the same default/non-default proportions in every split, which is important for imbalanced credit-risk data.

In [ ]:
def make_splits(X, y, protected):
    idx = np.arange(len(y))
    dev_idx, test_idx = train_test_split(
        idx, test_size=TEST_SIZE, stratify=y.iloc[idx], random_state=RANDOM_STATE
    )
    train_idx, cal_idx = train_test_split(
        dev_idx, test_size=CALIBRATION_SIZE_WITHIN_TRAIN,
        stratify=y.iloc[dev_idx], random_state=RANDOM_STATE
    )
    def take(obj, inds):
        return obj.iloc[inds].reset_index(drop=True)
    return {
        "X_train": take(X, train_idx), "y_train": take(y, train_idx), "A_train": take(protected, train_idx),
        "X_cal": take(X, cal_idx), "y_cal": take(y, cal_idx), "A_cal": take(protected, cal_idx),
        "X_test": take(X, test_idx), "y_test": take(y, test_idx), "A_test": take(protected, test_idx),
    }

for name, obj in DATASETS.items():
    obj["split"] = make_splits(obj["X"], obj["y"], obj["protected"])
    s=obj["split"]
    print(f"{name}: train={len(s['y_train']):,}, calibration={len(s['y_cal']):,}, test={len(s['y_test']):,}")
    print("  Test default rate:", f"{s['y_test'].mean():.2%}")

## 7. Preprocessing Pipelines and Model Definitions

### Model rationale from the proposal
- **Logistic Regression:** transparent and interpretable baseline.
- **Random Forest:** captures nonlinear relationships and interactions.
- **XGBoost:** strong boosting model commonly used for structured financial data.

All preprocessing is learned from training data only. The model objects are intentionally kept simple enough to explain in a viva.

In [ ]:
def make_preprocessor(X):
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", encoder)
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ], remainder="drop")


def model_factories(y_train):
    # No synthetic rebalancing is performed. Metrics explicitly address class imbalance.
    return {
        "Logistic Regression": lambda: LogisticRegression(
            max_iter=250, solver="liblinear", random_state=RANDOM_STATE
        ),
        "Random Forest": lambda: RandomForestClassifier(
            n_estimators=70, max_depth=12, min_samples_leaf=3,
            random_state=RANDOM_STATE, n_jobs=-1
        ),
        "XGBoost": lambda: XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            eval_metric="logloss", random_state=RANDOM_STATE,
            n_jobs=-1, tree_method="hist"
        )
    }


def make_model_pipeline(X_train, estimator):
    return Pipeline([
        ("preprocess", make_preprocessor(X_train)),
        ("model", estimator)
    ])

## 8. Baseline Model Training and Predictive Performance

**What are we doing?** Training all three model families on each dataset and evaluating only on the held-out test set.

**Output meaning.**
- **Accuracy:** overall proportion correct.
- **Precision:** among predicted defaults, how many actually defaulted.
- **Recall:** among actual defaults, how many were detected.
- **F1:** balance between Precision and Recall.
- **ROC-AUC:** ranking ability across thresholds; useful when classes are imbalanced.

In [ ]:
def performance_metrics(y_true, y_pred, y_prob=None):
    result = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }
    result["ROC_AUC"] = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    return result

baseline_models = {}
baseline_rows = []

for dataset_name, obj in DATASETS.items():
    print("\n" + "="*80 + f"\nBASELINE MODELS — {dataset_name}\n" + "="*80)
    s = obj["split"]
    baseline_models[dataset_name] = {}
    factories = model_factories(s["y_train"])
    for model_name, factory in factories.items():
        print("Training", model_name)
        pipe = make_model_pipeline(s["X_train"], factory())
        pipe.fit(s["X_train"], s["y_train"])
        pred = pipe.predict(s["X_test"]).astype(int)
        prob = pipe.predict_proba(s["X_test"])[:,1]
        metrics = performance_metrics(s["y_test"], pred, prob)
        baseline_models[dataset_name][model_name] = {"model":pipe, "pred":pred, "prob":prob}
        baseline_rows.append({"Dataset":dataset_name,"Model":model_name,"Method":"Baseline", **metrics})
        print({k:round(v,4) for k,v in metrics.items()})

baseline_performance = pd.DataFrame(baseline_rows)
display(baseline_performance.round(4))

for dataset_name in baseline_performance["Dataset"].unique():
    part = baseline_performance[baseline_performance.Dataset==dataset_name]
    best_auc = part.loc[part.ROC_AUC.idxmax()]
    best_f1 = part.loc[part.F1.idxmax()]
    print(f"{dataset_name}: highest ROC-AUC = {best_auc.Model} ({best_auc.ROC_AUC:.3f}); highest F1 = {best_f1.Model} ({best_f1.F1:.3f}).")

In [ ]:
# Confusion matrices — one figure per dataset, three model panels.
for dataset_name, obj in DATASETS.items():
    s=obj["split"]
    fig, axes = plt.subplots(1,3,figsize=(14,4))
    for ax,(model_name,res) in zip(axes, baseline_models[dataset_name].items()):
        ConfusionMatrixDisplay.from_predictions(s["y_test"], res["pred"], ax=ax, colorbar=False)
        ax.set_title(model_name)
    fig.suptitle(f"{dataset_name}: Baseline Confusion Matrices")
    plt.tight_layout(); plt.show()
    print("Interpretation: Compare false negatives carefully because an undetected default can be financially costly. F1 and Recall should be considered alongside overall Accuracy.")

# ROC curves — one figure per dataset.
for dataset_name, obj in DATASETS.items():
    s=obj["split"]
    fig, ax = plt.subplots(figsize=(6,5))
    for model_name,res in baseline_models[dataset_name].items():
        fpr,tpr,_=roc_curve(s["y_test"],res["prob"])
        auc=roc_auc_score(s["y_test"],res["prob"])
        ax.plot(fpr,tpr,label=f"{model_name} (AUC={auc:.3f})")
    ax.plot([0,1],[0,1],linestyle="--")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{dataset_name}: Baseline ROC Curves"); ax.legend()
    plt.tight_layout(); plt.show()
    print("Interpretation: The curve nearest the upper-left corner and the highest ROC-AUC indicates the strongest ranking performance on this dataset.")

## 9. Fairness Metrics

### Demographic Parity Difference (DPD)
Measures the largest difference in **favourable decision rates** between protected groups. A value of **0** indicates parity; larger absolute magnitude indicates greater disparity.

### Equalized Odds Difference (EOD)
Measures the largest group difference in error-sensitive rates conditional on the true outcome. A value of **0** is ideal; larger values indicate that prediction errors are distributed less equally across groups.

**Important:** Fairness metrics are diagnostic measures, not automatic legal findings of discrimination. The proposal itself notes that fairness cannot be represented by a single metric, so DPD and EOD are interpreted together.

In [ ]:
def fairness_metrics(y_true_default, y_pred_default, sensitive):
    # Remove only observations whose protected group is unavailable for this specific audit.
    sensitive = pd.Series(sensitive).reset_index(drop=True)
    y_true_default = pd.Series(np.asarray(y_true_default)).reset_index(drop=True)
    y_pred_default = pd.Series(np.asarray(y_pred_default)).reset_index(drop=True)
    valid = sensitive.notna()
    a = sensitive[valid]
    y_fav = 1 - y_true_default[valid]     # 1 = favourable / non-default
    pred_fav = 1 - y_pred_default[valid] # 1 = favourable / approved
    if a.nunique() < 2:
        return {"DPD":np.nan,"EOD":np.nan,"Max_Fairness_Gap":np.nan,"Group_Approval_Rates":{}}
    dpd = demographic_parity_difference(y_fav, pred_fav, sensitive_features=a)
    eod = equalized_odds_difference(y_fav, pred_fav, sensitive_features=a)
    approval = pd.DataFrame({"A":a,"approved":pred_fav}).groupby("A", observed=True)["approved"].mean().to_dict()
    return {
        "DPD":float(dpd),
        "EOD":float(eod),
        "Max_Fairness_Gap":float(max(abs(dpd),abs(eod))),
        "Group_Approval_Rates":approval
    }

baseline_fairness_rows=[]
for dataset_name,obj in DATASETS.items():
    s=obj["split"]
    for model_name,res in baseline_models[dataset_name].items():
        for attr in s["A_test"].columns:
            fm=fairness_metrics(s["y_test"],res["pred"],s["A_test"][attr])
            baseline_fairness_rows.append({
                "Dataset":dataset_name,"Model":model_name,"Method":"Baseline","Protected_Attribute":attr,
                "DPD":fm["DPD"],"EOD":fm["EOD"],"Max_Fairness_Gap":fm["Max_Fairness_Gap"],
                "Group_Approval_Rates":fm["Group_Approval_Rates"]
            })

baseline_fairness=pd.DataFrame(baseline_fairness_rows)
display(baseline_fairness.drop(columns="Group_Approval_Rates").round(4))

for _,r in baseline_fairness.iterrows():
    print(f"{r.Dataset} | {r.Model} | {r.Protected_Attribute}: DPD={r.DPD:.3f}, EOD={r.EOD:.3f}. "
          f"The larger gap is {r.Max_Fairness_Gap:.3f}; values closer to zero indicate stronger parity.")

In [ ]:
if SHOW_EXTRA_PLOTS:
    # Visual comparison of baseline fairness gaps.
    for dataset_name in baseline_fairness.Dataset.unique():
        p=baseline_fairness[baseline_fairness.Dataset==dataset_name].copy()
        labels=p["Model"]+"\n"+p["Protected_Attribute"]
        x=np.arange(len(p)); width=0.38
        fig,ax=plt.subplots(figsize=(10,5))
        ax.bar(x-width/2,p["DPD"].abs(),width,label="|DPD|")
        ax.bar(x+width/2,p["EOD"].abs(),width,label="EOD")
        ax.set_xticks(x); ax.set_xticklabels(labels,rotation=30,ha="right")
        ax.set_ylabel("Fairness gap (0 = ideal)")
        ax.set_title(f"{dataset_name}: Baseline Fairness by Model and Protected Attribute")
        ax.legend(); plt.tight_layout(); plt.show()
        worst=p.loc[p.Max_Fairness_Gap.idxmax()]
        print(f"Interpretation: The largest baseline fairness gap in {dataset_name} occurs for {worst.Model} on {worst.Protected_Attribute} ({worst.Max_Fairness_Gap:.3f}).")


## 10. Reweighing (Pre-processing Bias Mitigation)

**What are we doing?** Assigning training weights so protected-group/outcome combinations are represented according to what would be expected if protected group and outcome were independent.

For group `A=a` and target `Y=y`:

\[
w(a,y) = rac{P(A=a)P(Y=y)}{P(A=a,Y=y)}
\]

**Why appropriate?** It changes the *importance* of real training observations without fabricating or deleting observations. The test set is never reweighted.

**Viva point.** Reweighing is a pre-processing method: it acts before model prediction and can therefore be applied to Logistic Regression, Random Forest and XGBoost using sample weights.

In [ ]:
def reweighing_weights(y, sensitive):
    y=pd.Series(np.asarray(y)).reset_index(drop=True)
    a=pd.Series(sensitive).reset_index(drop=True)
    weights=np.ones(len(y),dtype=float)
    valid=a.notna()
    n=valid.sum()
    if n==0 or a[valid].nunique()<2:
        return weights
    p_a=a[valid].value_counts(normalize=True,dropna=False)
    p_y=y[valid].value_counts(normalize=True,dropna=False)
    joint=pd.DataFrame({"A":a[valid],"Y":y[valid]}).value_counts(normalize=True)
    for group in p_a.index:
        for outcome in p_y.index:
            mask=valid & (a==group) & (y==outcome)
            denom=joint.get((group,outcome),0)
            if denom>0:
                weights[mask.to_numpy()] = (p_a[group]*p_y[outcome])/denom
    return weights

reweighed_models={}
reweigh_perf_rows=[]
reweigh_fair_rows=[]

for dataset_name,obj in DATASETS.items():
    s=obj["split"]
    reweighed_models[dataset_name]={}
    for attr in s["A_train"].columns:
        weights=reweighing_weights(s["y_train"],s["A_train"][attr])
        print(f"\n{dataset_name} — Reweighing for {attr}: weight range {weights.min():.3f} to {weights.max():.3f}")
        reweighed_models[dataset_name][attr]={}
        for model_name,factory in model_factories(s["y_train"]).items():
            pipe=make_model_pipeline(s["X_train"],factory())
            pipe.fit(s["X_train"],s["y_train"],model__sample_weight=weights)
            pred=pipe.predict(s["X_test"]).astype(int)
            prob=pipe.predict_proba(s["X_test"])[:,1]
            pm=performance_metrics(s["y_test"],pred,prob)
            fm=fairness_metrics(s["y_test"],pred,s["A_test"][attr])
            reweighed_models[dataset_name][attr][model_name]={"model":pipe,"pred":pred,"prob":prob}
            reweigh_perf_rows.append({"Dataset":dataset_name,"Model":model_name,"Method":"Reweighing","Protected_Attribute":attr,**pm})
            reweigh_fair_rows.append({"Dataset":dataset_name,"Model":model_name,"Method":"Reweighing","Protected_Attribute":attr,
                                      "DPD":fm["DPD"],"EOD":fm["EOD"],"Max_Fairness_Gap":fm["Max_Fairness_Gap"],
                                      "Group_Approval_Rates":fm["Group_Approval_Rates"]})

reweigh_performance=pd.DataFrame(reweigh_perf_rows)
reweigh_fairness=pd.DataFrame(reweigh_fair_rows)
display(reweigh_performance.round(4))
display(reweigh_fairness.drop(columns="Group_Approval_Rates").round(4))

## 11. Equalized Odds Post-processing

**What are we doing?** Keeping each already-trained baseline model fixed and learning group-aware decision thresholds on the **calibration subset** to reduce unequal error rates.

**Why appropriate?** It is a true post-processing method: the underlying model is not retrained. It directly targets Equalized Odds, matching the methodology in the proposal.

**Important limitation.** Post-processing requires the protected attribute at decision time. This has operational/legal implications and should be discussed rather than ignored.

In [ ]:
eo_perf_rows=[]
eo_fair_rows=[]
eo_objects={}

for dataset_name,obj in DATASETS.items():
    s=obj["split"]
    eo_objects[dataset_name]={}
    for attr in s["A_cal"].columns:
        # Equalized-odds optimizer needs usable group labels and enough observations.
        cal_valid=s["A_cal"][attr].notna()
        test_valid=s["A_test"][attr].notna()
        if s["A_cal"].loc[cal_valid,attr].nunique()<2:
            print(f"Skipping {dataset_name}/{attr}: fewer than two valid calibration groups.")
            continue
        eo_objects[dataset_name][attr]={}
        for model_name,res in baseline_models[dataset_name].items():
            base_model=res["model"]
            try:
                post=ThresholdOptimizer(
                    estimator=base_model,
                    constraints="equalized_odds",
                    objective="balanced_accuracy_score",
                    predict_method="predict_proba",
                    prefit=True
                )
                post.fit(
                    s["X_cal"].loc[cal_valid].reset_index(drop=True),
                    s["y_cal"].loc[cal_valid].reset_index(drop=True),
                    sensitive_features=s["A_cal"].loc[cal_valid,attr].reset_index(drop=True)
                )
                # Evaluate on the same valid protected-group subset for both performance and fairness.
                Xev=s["X_test"].loc[test_valid].reset_index(drop=True)
                yev=s["y_test"].loc[test_valid].reset_index(drop=True)
                Aev=s["A_test"].loc[test_valid,attr].reset_index(drop=True)
                pred=post.predict(Xev,sensitive_features=Aev,random_state=RANDOM_STATE).astype(int)
                # ThresholdOptimizer provides decisions, not a single post-processed probability score.
                pm=performance_metrics(yev,pred,None)
                fm=fairness_metrics(yev,pred,Aev)
                eo_objects[dataset_name][attr][model_name]={"postprocessor":post,"pred":pred}
                eo_perf_rows.append({"Dataset":dataset_name,"Model":model_name,"Method":"Equalized Odds","Protected_Attribute":attr,**pm})
                eo_fair_rows.append({"Dataset":dataset_name,"Model":model_name,"Method":"Equalized Odds","Protected_Attribute":attr,
                                     "DPD":fm["DPD"],"EOD":fm["EOD"],"Max_Fairness_Gap":fm["Max_Fairness_Gap"],
                                     "Group_Approval_Rates":fm["Group_Approval_Rates"]})
            except Exception as e:
                print(f"Equalized Odds could not be applied to {dataset_name}/{model_name}/{attr}: {e}")
                print("Limitation recorded rather than manipulating the data to force the method to work.")

eo_performance=pd.DataFrame(eo_perf_rows)
eo_fairness=pd.DataFrame(eo_fair_rows)
display(eo_performance.round(4))
display(eo_fairness.drop(columns="Group_Approval_Rates").round(4))

## 12. Consolidated Baseline → Reweighing → Equalized Odds Comparison

This section directly addresses the core methodological comparison required in the proposal.

Because Equalized-Odds post-processing creates binary decisions rather than a single comparable probability score, **ROC-AUC is not used to judge the post-processed decisions**. Accuracy, Precision, Recall and F1 remain directly comparable. For the accuracy–fairness trade-off, F1 is therefore used as the main common performance metric across all three methods, with baseline/Reweighing ROC-AUC retained as supplementary evidence.

In [ ]:
# Expand baseline performance by protected attribute so it can be matched fairly with each mitigation run.
base_perf_expanded=[]
for _,r in baseline_performance.iterrows():
    attrs=DATASETS[r.Dataset]["split"]["A_test"].columns
    for attr in attrs:
        base_perf_expanded.append({**r.to_dict(),"Protected_Attribute":attr})
base_perf_expanded=pd.DataFrame(base_perf_expanded)

all_perf=pd.concat([base_perf_expanded,reweigh_performance,eo_performance],ignore_index=True,sort=False)
all_fair=pd.concat([baseline_fairness,reweigh_fairness,eo_fairness],ignore_index=True,sort=False)

comparison=all_perf.merge(
    all_fair.drop(columns="Group_Approval_Rates"),
    on=["Dataset","Model","Method","Protected_Attribute"], how="inner"
)
comparison["Abs_DPD"]=comparison["DPD"].abs()
comparison["Abs_EOD"]=comparison["EOD"].abs()
comparison["Fairness_Gap"]=comparison[["Abs_DPD","Abs_EOD"]].max(axis=1)

display(comparison.sort_values(["Dataset","Protected_Attribute","Model","Method"]).round(4))

In [ ]:
# Calculate changes relative to the correct baseline for the same dataset/model/protected attribute.
base_reference=comparison[comparison.Method=="Baseline"][
    ["Dataset","Model","Protected_Attribute","Accuracy","Precision","Recall","F1","ROC_AUC","Abs_DPD","Abs_EOD","Fairness_Gap"]
].rename(columns={c:"Baseline_"+c for c in ["Accuracy","Precision","Recall","F1","ROC_AUC","Abs_DPD","Abs_EOD","Fairness_Gap"]})

change=comparison.merge(base_reference,on=["Dataset","Model","Protected_Attribute"],how="left")
for metric in ["Accuracy","Precision","Recall","F1","ROC_AUC","Abs_DPD","Abs_EOD","Fairness_Gap"]:
    change[f"Delta_{metric}"]=change[metric]-change[f"Baseline_{metric}"]

# Negative fairness delta = improvement because gap moved closer to zero.
cols=["Dataset","Model","Protected_Attribute","Method","F1","Fairness_Gap","Delta_F1","Delta_Fairness_Gap","Abs_DPD","Abs_EOD"]
display(change[cols].sort_values(["Dataset","Protected_Attribute","Model","Method"]).round(4))

for _,r in change[change.Method!="Baseline"].iterrows():
    fairness_direction="improved" if r.Delta_Fairness_Gap < 0 else "worsened" if r.Delta_Fairness_Gap > 0 else "did not change"
    f1_direction="increased" if r.Delta_F1 > 0 else "decreased" if r.Delta_F1 < 0 else "did not change"
    print(f"{r.Dataset} | {r.Model} | {r.Protected_Attribute} | {r.Method}: fairness {fairness_direction} "
          f"(Δ gap={r.Delta_Fairness_Gap:+.3f}) while F1 {f1_direction} (Δ F1={r.Delta_F1:+.3f}).")

In [ ]:
# Compact before-vs-after visualisations (kept small so Kaggle versions save faster).
plot_data = (comparison.groupby(["Method"], as_index=False)
             .agg(Mean_F1=("F1","mean"), Mean_Fairness_Gap=("Fairness_Gap","mean")))

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(plot_data["Method"], plot_data["Mean_Fairness_Gap"])
ax.set_ylabel("Mean fairness gap (lower is better)")
ax.set_title("Overall Fairness: Baseline vs Mitigation")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(plot_data["Method"], plot_data["Mean_F1"])
ax.set_ylabel("Mean F1 score")
ax.set_title("Overall Predictive Performance: Baseline vs Mitigation")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

print("Interpretation: compare the lower fairness gap with the F1 score to see whether mitigation reduced disparity at a performance cost.")


## 13. Accuracy–Fairness Trade-off

A lower fairness gap is desirable, while a higher F1 is desirable. Rather than hiding this conflict inside an arbitrary weighted score, the notebook first plots the two dimensions directly.

For a concise ranking needed for the final research conclusion, it then uses a transparent **rank-sum** within each dataset and protected attribute:

- rank 1 = highest F1
- rank 1 = lowest fairness gap
- **Balance Rank Sum = Performance Rank + Fairness Rank**

Lower is better. This rank-sum is a descriptive decision aid, **not** a new fairness metric or legal standard.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for method, grp in comparison.groupby("Method"):
    ax.scatter(grp["Fairness_Gap"], grp["F1"], label=method, alpha=0.7)
ax.set_xlabel("Fairness gap: max(|DPD|, EOD) — lower is better")
ax.set_ylabel("F1 for default class — higher is better")
ax.set_title("Performance–Fairness Trade-off Across All Datasets")
ax.legend(); plt.tight_layout(); plt.show()
print("Interpretation: points toward the upper-left indicate a stronger empirical fairness–performance balance.")

ranked=comparison.copy()
ranked["Performance_Rank"]=ranked.groupby(["Dataset","Protected_Attribute"])["F1"].rank(ascending=False,method="min")
ranked["Fairness_Rank"]=ranked.groupby(["Dataset","Protected_Attribute"])["Fairness_Gap"].rank(ascending=True,method="min")
ranked["Balance_Rank_Sum"]=ranked["Performance_Rank"]+ranked["Fairness_Rank"]

best_balance=(ranked.sort_values(["Dataset","Protected_Attribute","Balance_Rank_Sum","Fairness_Gap"])
              .groupby(["Dataset","Protected_Attribute"],as_index=False).first())
display(best_balance[["Dataset","Protected_Attribute","Model","Method","F1","Fairness_Gap","Balance_Rank_Sum"]].round(4))


## 14. Cross-Dataset Comparison

This section asks whether findings are consistent across datasets rather than relying on one population. Because dataset sizes, variables and base default rates differ, absolute performance should not be interpreted as if the datasets were identical experiments.

In [ ]:
cross_summary=(comparison.groupby(["Dataset","Model","Method"],as_index=False)
               .agg(Mean_F1=("F1","mean"),
                    Mean_Fairness_Gap=("Fairness_Gap","mean"),
                    Mean_Abs_DPD=("Abs_DPD","mean"),
                    Mean_Abs_EOD=("Abs_EOD","mean")))
display(cross_summary.round(4))

# Average ranks across every dataset/protected attribute combination.
overall_rank=(ranked.groupby(["Model","Method"],as_index=False)
              .agg(Mean_Performance_Rank=("Performance_Rank","mean"),
                   Mean_Fairness_Rank=("Fairness_Rank","mean"),
                   Mean_Balance_Rank=("Balance_Rank_Sum","mean"),
                   Mean_F1=("F1","mean"),
                   Mean_Fairness_Gap=("Fairness_Gap","mean"))
              .sort_values(["Mean_Balance_Rank","Mean_Fairness_Gap"]))
display(overall_rank.round(4))

# Cross-dataset chart omitted in FAST notebook to reduce saved-version size.

winner=overall_rank.iloc[0]
print(f"Interpretation: By the pre-declared rank-sum decision aid, the strongest average balance is {winner.Model} with {winner.Method} (mean balance rank {winner.Mean_Balance_Rank:.2f}).")

## 15. Data-Driven Answers to the Research Questions

The following text is generated from the calculated results. No numerical conclusion is hard-coded in advance.

In [ ]:
def research_question_conclusions(comparison, change, overall_rank):
    print("RESEARCH QUESTION 1: Do machine-learning credit scoring models produce unfair outcomes across demographic groups?\n")
    baseline=comparison[comparison.Method=="Baseline"]
    worst=baseline.loc[baseline.Fairness_Gap.idxmax()]
    best=baseline.loc[baseline.Fairness_Gap.idxmin()]
    print(f"Across the baseline audits, fairness gaps were not identical across groups. The largest observed baseline gap was {worst.Fairness_Gap:.3f} "
          f"for {worst.Model} on {worst.Dataset}/{worst.Protected_Attribute}; the smallest was {best.Fairness_Gap:.3f}. "
          "This provides empirical evidence of differing group outcomes/errors, although the metrics alone are not a legal finding of discrimination.\n")

    print("RESEARCH QUESTION 2: Which model demonstrates the best balance between fairness and predictive performance?\n")
    w=overall_rank.iloc[0]
    print(f"Using the declared equal-weight rank-sum of F1 performance and fairness gap, {w.Model} with {w.Method} achieved the strongest average cross-dataset balance "
          f"(mean balance rank {w.Mean_Balance_Rank:.2f}, mean F1 {w.Mean_F1:.3f}, mean fairness gap {w.Mean_Fairness_Gap:.3f}). "
          "This conclusion is specific to the rank-based decision rule and should be considered alongside dataset-specific results.\n")

    print("RESEARCH QUESTION 3: How effective are Reweighing and Equalized Odds Post-processing?\n")
    m=change[change.Method!="Baseline"].groupby("Method").agg(
        Mean_Delta_Fairness=("Delta_Fairness_Gap","mean"),
        Mean_Delta_F1=("Delta_F1","mean")
    )
    display(m.round(4))
    for method,r in m.iterrows():
        effect="reduced" if r.Mean_Delta_Fairness<0 else "increased"
        perf="improved" if r.Mean_Delta_F1>0 else "reduced"
        print(f"{method} {effect} the fairness gap by {abs(r.Mean_Delta_Fairness):.3f} on average and {perf} F1 by {abs(r.Mean_Delta_F1):.3f} on average.")
    print()

    print("RESEARCH QUESTION 4: What is the relationship between fairness, explainability and predictability?\n")
    baseline_perf=comparison[comparison.Method=="Baseline"].groupby("Model").agg(Mean_F1=("F1","mean"),Mean_Gap=("Fairness_Gap","mean")).sort_values("Mean_F1",ascending=False)
    display(baseline_perf.round(4))
    print("The predictive-performance and fairness components above are data-driven. Explainability, however, was not measured by a numerical instrument in the proposal. "
          "Therefore it would be methodologically incorrect to invent an explainability score. Logistic Regression is structurally the most transparent of the three model families; "
          "Random Forest and especially boosted-tree ensembles are more complex. The appropriate conclusion is therefore a qualitative comparison of model transparency alongside the calculated F1 and fairness gaps, rather than a fabricated explainability statistic.")

research_question_conclusions(comparison,change,overall_rank)

## 16. Research Objective Conclusions

This section maps the calculated evidence back to the five objectives in the proposal.

In [ ]:
print("OBJECTIVE 1 — Assess fairness of Logistic Regression, Random Forest and XGBoost")
print("Completed: all three models were trained on all three uploaded datasets and evaluated on held-out test data.")

display(baseline_performance.groupby("Model")[["Accuracy","Precision","Recall","F1","ROC_AUC"]].mean().round(4))

print("\nOBJECTIVE 2 — Measure bias using DPD and EOD")
display(baseline_fairness.groupby(["Model","Protected_Attribute"])[["DPD","EOD","Max_Fairness_Gap"]].mean().round(4))

print("\nOBJECTIVE 3 — Apply and compare Reweighing and Equalized Odds")
mitigation_summary=change[change.Method!="Baseline"].groupby("Method")[["Delta_F1","Delta_Fairness_Gap"]].mean()
display(mitigation_summary.round(4))

print("\nOBJECTIVE 4 — Examine the accuracy–fairness trade-off")
display(overall_rank.round(4))

print("\nOBJECTIVE 5 — Explore implications for UK financial regulation and responsibility")
print("The empirical audit demonstrates why credit models should be evaluated for both predictive quality and group disparities. "
      "Protected-feature removal alone does not guarantee parity because proxy variables can remain. Any deployment of group-aware post-processing also raises governance questions because protected information may be required at decision time. "
      "The regulatory discussion should therefore focus on transparent monitoring, documented thresholds, human oversight, data quality, and explainability rather than treating one fairness metric as sufficient.")

## 17. Key Findings and Viva Interpretation

Run the next cell after the whole notebook. It automatically converts the calculated results into concise viva-ready findings.

In [ ]:
def viva_summary():
    print("KEY FINDINGS AND VIVA INTERPRETATION")
    print("="*80)
    print("1. DATA PREPARATION")
    print("The three datasets were not forced into a common raw schema. Each dataset kept its own variables and target was standardised only semantically: 1=default and 0=non-default.")
    print("Missing numerical values were median-imputed and categorical values received an explicit Missing category, with all transformations fitted only on training data.")
    print("Protected attributes were separated from predictors so they could be audited without directly driving model predictions.")

    print("\n2. PREDICTIVE PERFORMANCE")
    for ds in baseline_performance.Dataset.unique():
        p=baseline_performance[baseline_performance.Dataset==ds]
        r=p.loc[p.ROC_AUC.idxmax()]
        print(f"{ds}: {r.Model} had the highest baseline ROC-AUC ({r.ROC_AUC:.3f}); its F1 was {r.F1:.3f}.")

    print("\n3. BASELINE FAIRNESS")
    for ds in baseline_fairness.Dataset.unique():
        p=baseline_fairness[baseline_fairness.Dataset==ds]
        r=p.loc[p.Max_Fairness_Gap.idxmax()]
        print(f"{ds}: the largest baseline fairness gap was {r.Max_Fairness_Gap:.3f} for {r.Model} on {r.Protected_Attribute}.")

    print("\n4. MITIGATION")
    m=change[change.Method!="Baseline"].groupby("Method").agg(dfair=("Delta_Fairness_Gap","mean"),df1=("Delta_F1","mean"))
    for method,r in m.iterrows():
        print(f"{method}: mean fairness-gap change {r.dfair:+.3f}; mean F1 change {r.df1:+.3f}. Negative fairness change means improvement.")

    print("\n5. BEST PERFORMANCE–FAIRNESS BALANCE")
    w=overall_rank.iloc[0]
    print(f"The rank-based cross-dataset comparison selected {w.Model} with {w.Method} as the strongest average balance (mean balance rank {w.Mean_Balance_Rank:.2f}).")

    print("\n6. LIMITATIONS TO DEFEND")
    print("- The proposal originally named FCA Financial Lives, but the third supplied dataset is Taiwan; the dissertation must accurately state what was analysed.")
    print("- Age bands are analytical audit groups, not legal definitions.")
    print("- DPD and EOD measure different fairness concepts and can conflict; neither is a standalone legal test.")
    print("- Equalized-Odds post-processing can require protected-group membership at decision time, creating governance considerations.")
    print("- Explainability was not quantitatively measured; it is discussed using model structure rather than an invented score.")
    print("- Results are dataset-specific and observational; they do not prove causal discrimination.")

viva_summary()

## 18. Export Final Tables for the Dissertation

These CSV outputs make it easier to use the exact calculated numbers in the written dissertation and prevent transcription errors.

In [ ]:
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

baseline_performance.to_csv(os.path.join(OUTPUT_DIR,"01_baseline_performance.csv"),index=False)
baseline_fairness.to_csv(os.path.join(OUTPUT_DIR,"02_baseline_fairness.csv"),index=False)
reweigh_performance.to_csv(os.path.join(OUTPUT_DIR,"03_reweighing_performance.csv"),index=False)
reweigh_fairness.to_csv(os.path.join(OUTPUT_DIR,"04_reweighing_fairness.csv"),index=False)
eo_performance.to_csv(os.path.join(OUTPUT_DIR,"05_equalized_odds_performance.csv"),index=False)
eo_fairness.to_csv(os.path.join(OUTPUT_DIR,"06_equalized_odds_fairness.csv"),index=False)
comparison.to_csv(os.path.join(OUTPUT_DIR,"07_full_comparison.csv"),index=False)
change.to_csv(os.path.join(OUTPUT_DIR,"08_before_after_changes.csv"),index=False)
overall_rank.to_csv(os.path.join(OUTPUT_DIR,"09_cross_dataset_balance.csv"),index=False)

print("Saved result tables to:", OUTPUT_DIR)

# Overall Methodological Conclusion

After executing all cells, use the generated research-question conclusions and **Key Findings and Viva Interpretation** as the numerical basis for the dissertation conclusion. Do not copy claims from another run or another dataset version.

The core defensible methodological story is:

1. Three real credit-risk datasets were inspected separately rather than forced into one schema.
2. Targets were aligned to the same semantic direction (1=default) while preserving observations.
3. Protected attributes were withheld from prediction and retained for auditing.
4. Logistic Regression, Random Forest and XGBoost were compared on the same held-out test logic.
5. Bias was audited using both DPD and EOD because fairness is multidimensional.
6. Reweighing was evaluated as a pre-processing method and Equalized Odds as post-processing.
7. The final conclusion considers **both** predictive performance and fairness instead of declaring the most accurate model automatically best.
8. Limitations are stated explicitly where the proposal does not support a stronger claim, particularly for quantitative explainability and the replacement of FCA Financial Lives with the Taiwan dataset.